# Calibrating WFI Exposures with RomanCal

## Kernel Information and Read-Only Status

To run this notebook, please select "Roman Research Nexus {VERSION}" kernel at the top right of your window. For example "Roman Research Nexus 2026.2".

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.
    

## Introduction
The purpose of this notebook is to calibrate Level 1 (L1; uncalibrated ramp cube) data with the Roman WFI science calibration pipeline RomanCal (Python package name `romancal`) to produce Level 2 (L2; calibrated rate image) exposure level data. To learn more, please visit the [RDox pages on the Exposure Level Pipeline](https://roman-docs.stsci.edu/data-handbook/roman-wfi-data-pipelines/exposure-level-pipeline) (ELP). We also discuss calibration reference files including how to access and examine them and how to run the pipeline with custom reference files.

Details about the Roman data levels, including file naming conventions and file array names and data types, can be found in the RDox article [Data Levels and Products](https://roman-docs.stsci.edu/data-handbook-home/wfi-data-format/data-levels-and-products). 

A L1 file contains uncalibrated ramps in units of Data Numbers (DN).  L1 files are three-dimensional data cubes, one dimension for time and two dimensions for image coordinates, that are shaped as  arrays with (N resultants, 4096 image rows, 4096 image columns). For a given pixel, a resultant contains either one read or the arithmetic mean of multiple reads. 

L2 WFI files are calibrated rate images in instrumental units of DN / second.  They are two-dimensional arrays shaped as (4088 image rows, 4088 image columns). Note the smaller image size of L2 files, which is due to the removal of the 4-pixel border of reference pixels around the image during pipeline processing.

### Local Run Settings

If you want to run the notebook in your local machine, refer to the information in [local installation](../../markdown/local-run.md) instructions before proceeding with the notebook. The instructions provide inportant information about setting up your environment and installing dependnecies.

## Imports
Libraries used:
- *astropy* for coordinates manipulation and image normalization
- *copy* for making copies of Python objects
- *crds* for access to calibration reference files
- *matplotlib* and *mpl_toolkits* for plotting images
- *numpy* for array manipulation
- *romancal* for running the Roman WFI science data pipeline
- *roman_datamodels* for opening Roman WFI ASDF files
- *asdf* for opening Roman WFI ASDF files
- *os* for operating system functions
- *s3fs* for streaming files from an AWS S3 bucket

In [1]:
import os

from astropy.coordinates import SkyCoord
from astropy.visualization import simple_norm
import copy

import matplotlib.pyplot as plt
from matplotlib import colors, colormaps as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import roman_datamodels as rdm
import s3fs

### The Calibration Reference Data System (CRDS)

The Roman ELP, which corrects L1 images for detector-level effects to produce L2 images, uses calibration reference and parameter files from the [CRDS](https://roman-crds.stsci.edu/static/users_guide/overview.html). These reference files, developed and validated by STScI’s Science Operations Center, are continually updated as new WFI data become available. CRDS assigns the most appropriate reference file for each calibration step using metadata keywords and file-specific matching criteria. To use the best-available reference files for an observation, no action is needed as RomanCal will query for the best reference files for each calibration step in the pipeline.

In this tutorial, we will focus on the **[`crds`](https://roman-crds.stsci.edu/static/users_guide/index.html)** Python application programming interface (API). The [**CRDS** webserver](https://roman-crds.stsci.edu) can also be accessed to browse calibration reference files in a tabular interface. Note that there are multiple CRDS servers, though most users will interact with the Operations (OPS) instance. Please be sure to navigate to the correct webserver for the instance in which you are interested.

For more details, see the [RDox page on CRDS for Roman WFI](https://roman-docs.stsci.edu/data-handbook-home/accessing-wfi-data/crds-for-reference-files) and the [CRDS documentation](https://jwst-crds.stsci.edu/static/users_guide/web_site_use.html).

In [2]:
import crds

In [3]:
# Import romancal packages
import romancal
from romancal.pipeline import ExposurePipeline

## Tutorial Data

In this tutorial, we use L1 WFI data files simulated with Roman I-Sim. As an example, we take the output product from the [Roman I-Sim](../romanisim/romanisim.ipynb) tutorial notebook. If you have not run that simulation, the files are available in the Nexus S3 bucket. For more information on accessing these data, see the [Data Discovery and Access](../data_discovery_and_access/data_discovery_and_access.ipynb) tutorial.

## Running the ELP on L1 Data

To run the ELP on L1 data, you have two options:
1. **Basic:** Use `romancal.ExposurePipeline()` to run all steps.
2. **Advanced:** Run one or more individual steps.

### Basic Example: Full Pipeline
The input file for this example is a WFI L1 ASDF file. First, we check whether the file is already saved on disk (if the Roman I-Sim tutorial was run). If not, we stream the L1 file into memory (as a datamodel) from the Nexus S3 bucket:

In [4]:
l1_file = 'r0003201001001001004_0001_wfi01_f106_uncal.asdf'

if os.path.exists(l1_file):
    dm_l1 = rdm.open(l1_file)
else:
    s3_uri = asdf_dir_uri = 's3://stpubdata/roman/nexus/soc_simulations/tutorial_data/roman-2026.2/'
    fs = s3fs.S3FileSystem(anon=True)
    dm_l1 = rdm.open(fs.open(s3_uri + l1_file, 'rb'))

We begin by examining the data type using the `type()` function:

In [5]:
type(dm_l1)

roman_datamodels.datamodels._datamodels.ScienceRawModel

Reading the ASDF file with `roman_datamodels` returns a `ScienceRawModel` datamodel, which is the L1 file datamodel. At this point, we can use the `.info()` method on the data to look at the file contents:


In [6]:
dm_l1.info(max_rows=15) 

root (AsdfObject)
├─asdf_library (Software)
│ ├─author (str): The ASDF Developers
│ ├─homepage (str): http://github.com/asdf-format/asdf
│ ├─name (str): asdf
│ └─version (str): 5.3.1
├─history (AsdfDictNode)
│ └─extensions (AsdfListNode) ...
├─roman (WfiScienceRaw) # Level 1 (L1) Uncalibrated Roman Wide Field
Instrument (WFI) Ramp Cube

│ ├─meta (AsdfDictNode) ...
│ ├─data (NDArrayType) # Science Data (DN) ...
│ └─amp33 (NDArrayType) # Amplifier 33 Reference Pixel Data (DN) ...
└─romanisim (AsdfDictNode) ...
Some nodes not shown.


We can see that this L1 file was created with Roman I-Sim as it contains the "romanisim" block.

Next, we present a basic example of running the complete pipeline.

The optional `save_results` parameter determines whether the resulting L2 datamodel is saved as a file on your Nexus storage. Setting this parameter to `True` enables file saving. In this example, we retain the calibrated L2 datamodel in memory as the variable result without saving it locally.

In [7]:
result = ExposurePipeline.call(dm_l1, save_results=False)

2026-07-22 21:47:48,322 - stpipe.step - INFO - ExposurePipeline instance created.


2026-07-22 21:47:48,323 - stpipe.step - INFO - DQInitStep instance created.


2026-07-22 21:47:48,324 - stpipe.step - INFO - SaturationStep instance created.


2026-07-22 21:47:48,325 - stpipe.step - INFO - RefPixStep instance created.


2026-07-22 21:47:48,326 - stpipe.step - INFO - DarkDecayStep instance created.


...

2026-07-22 21:49:42,690 - romancal.assign_wcs.assign_wcs - INFO - S_REGION VALUES: POLYGON ICRS  270.934685535 -0.225734685 270.934628042 -0.102770924 270.809038204 -0.102466434 270.809760222 -0.225325584


2026-07-22 21:49:42,691 - romancal.assign_wcs.assign_wcs - INFO - Update S_REGION to POLYGON ICRS  270.934685535 -0.225734685 270.934628042 -0.102770924 270.809038204 -0.102466434 270.809760222 -0.225325584


2026-07-22 21:49:43,290 - stpipe.step - INFO - Step tweakreg done


2026-07-22 21:49:43,291 - romancal.pipeline.exposure_pipeline - INFO - Roman exposure calibration pipeline ending...


2026-07-22 21:49:43,867 - stpipe.step - INFO - Step ExposurePipeline done


In [8]:
type(result)

roman_datamodels.datamodels._datamodels.ImageModel

The output from the Exposure Pipeline is an `ImageModel` object, which serves as the datamodel for L2 files.

In addition, the pipeline created three other files in the working directory with names similar to the input L1 file. These files end with `*_cat.parquet`, `*_segm.asdf`, and `*_wcs.asdf`, corresponding to a Level 4 (L4) single-band source catalog, a L4 segmentation map, and a L1 updated WCS file, respectively. The L1 WCS file provides a L2-quality World Coordinate System (WCS) when working with a L1 file, which have a different number of pixels than L2 files, without updating the original L1 file on disk. More information about working with L4 products will be added in another future tutorial. **Note:** L4 products are still being validated and should be used with caution during development.

Optional parameters can also be passed to individual pipeline steps through the steps dictionary. For example, the code below demonstrates how to skip both the source catalog step and the step that aligns the image with the Gaia astrometric catalog (the TweakReg step, named after the software used to update the WCS). Other parameters can be configured in the same way; for details, see the [romancal documentation](https://roman-pipeline.readthedocs.io/en/latest/index.html).

In [9]:
result = ExposurePipeline.call(dm_l1, steps={'source_catalog': {'skip': True}, 'tweakreg': {'skip': True}})

2026-07-22 21:49:43,893 - stpipe.step - INFO - ExposurePipeline instance created.


2026-07-22 21:49:43,895 - stpipe.step - INFO - DQInitStep instance created.


2026-07-22 21:49:43,895 - stpipe.step - INFO - SaturationStep instance created.


2026-07-22 21:49:43,896 - stpipe.step - INFO - RefPixStep instance created.


2026-07-22 21:49:43,897 - stpipe.step - INFO - DarkDecayStep instance created.


...

2026-07-22 21:51:05,570 - stpipe.step - INFO - Step skipped.


2026-07-22 21:51:06,317 - stpipe.step - INFO - Step tweakreg running with args (<romancal.datamodels.library.ModelLibrary object at 0x7f4ccd3474d0>,).


2026-07-22 21:51:06,317 - stpipe.step - INFO - Step skipped.


2026-07-22 21:51:06,898 - romancal.pipeline.exposure_pipeline - INFO - Roman exposure calibration pipeline ending...


2026-07-22 21:51:07,480 - stpipe.step - INFO - Step ExposurePipeline done


At the end of the pipeline log messages, you will see that the SourceCatalog and TweakReg steps were skipped, as expected. Because the source catalog step was omitted, the L4 source catalog and segmentation map files were not regenerated. To verify the status of a step, you can also inspect the metadata of the output datamodel:

In [10]:
result.meta.cal_step.source_catalog

'SKIPPED'

More information on these steps can be found in the [Exposure Pipeline](https://roman-docs.stsci.edu/data-handbook-home/roman-data-pipelines/exposure-level-pipeline#ExposureLevelPipeline-PipelineStepDescriptions) article on RDox.

Note that the ramp-fitting step transforms the structure of the data — in other words, the data models before and after ramp fitting are intrinsically different. Therefore, steps following ramp fitting cannot be applied to data that has not undergone it, and similarly, steps preceding ramp fitting cannot be applied once the data has.

We can save our L2 datamodel to disk with the `.save()` method. The following commented command provides an example.

In [11]:
#result.save('my_roman_l2_file.asdf')

If you look at the file browser in the directory where you ran this tutorial, you should see a new file called "my_roman_l2_file.asdf". Note that you may need to wait a moment or manually refresh the file browser before it appears.

### Advanced Example: Running Individual Pipeline Steps

Now, for a more advanced use case, let's update the WCS based on the pointing information. For example, suppose we simulated an L1 file, processed it with the ELP, and now want to try shifting the pointing information and creating a new WCS to test the Gaia alignment. After editing any of the `meta.wcsinfo` values we wish to change, we can generate a new WCS by running the AssignWcsStep on our L2 ASDF file.

Let's start by reading in a fresh L2 file:

In [12]:
l2_file = 'r0003201001001001004_0001_wfi01_f106_cal.asdf'

if os.path.exists(l2_file):
    dm = rdm.open(l2_file)
    original_wcs = copy.deepcopy(dm.meta.wcs)
else:
    s3_uri = asdf_dir_uri = 's3://stpubdata/roman/nexus/soc_simulations/tutorial_data/roman-2026.2/'
    fs = s3fs.S3FileSystem(anon=True)
    dm = rdm.open(fs.open(s3_uri + l2_file, 'rb'))
    original_wcs = copy.deepcopy(dm.meta.wcs)

Let's take a quick look at the file we just opened:

In [13]:
dm.info()

root (AsdfObject)
├─asdf_library (Software)
│ ├─author (str): The ASDF Developers
│ ├─homepage (str): http://github.com/asdf-format/asdf
│ ├─name (str): asdf
│ └─version (str): 5.3.1
├─history (AsdfDictNode)
│ └─extensions (AsdfListNode) ...
└─roman (WfiImage) # Level 2 (L2) Calibrated Roman Wide Field Instrument (WFI) Rate Image.
  ├─meta (AsdfDictNode) ...
  ├─data (NDArrayType) # Science Data (DN/s) or (MJy/sr) ...
  ├─dq (NDArrayType) # Data Quality ...
  ├─err (NDArrayType) # Error (DN / s) or (MJy / sr) ...
  ├─var_poisson (NDArrayType) # Poisson Variance (DN^2/s^2) or (MJy^2/sr^2) ...
  ├─chisq (NDArrayType) # Chi-squared of Ramp Fit ...
  ├─dumo (NDArrayType) # Difference Uniform Minus Optimal (DN / s) or (MJy / sr) ...
  ├─amp33 (NDArrayType) # Amplifier 33 Reference Pixel Data (DN) ...
  ├─border_ref_pix_left (NDArrayType) # Left-Edge Border Reference Pixels (DN) ...
  ├─border_ref_pix_right (NDArrayType) # Right-Edge Border Reference Pixels (DN) ...
  ├─border_ref_pix_top (N

Before aligning the images with the Gaia coordinates, the WCS in an L2 file is populated using the telescope pointing information, resulting in the so-called "coarse WCS". The `meta.pointing` section of the metadata describes the spacecraft pointing, while the detector-dependent information used to construct the coarse WCS is contained in the `meta.wcsinfo` section. Although the values in `meta.pointing` and `meta.wcsinfo` are linked, the coarse WCS relies only on `meta.wcsinfo`. Let’s examine our `meta.wcsinfo` values:

In [14]:
dm.meta.wcsinfo

{'aperture_name': 'WFI01_FULL', 'dec_ref': -0.16439949970729792, 'ra_ref': 270.8719766359773, 'roll_ref': 59.99991238605931, 's_region': 'POLYGON ICRS  270.934685540 -0.225734681 270.934628038 -0.102770925 270.809038197 -0.102466445 270.809760224 -0.225325590', 'v2_ref': 1312.9491452484797, 'v3_ref': -1040.7853726755036, 'v3yangle': -60.0, 'vparity': -1}

Let's focus on the `ra_ref`, `dec_ref`, and `roll_ref` keywords. Let's first take a look at the descriptions of these fields:

In [15]:
print(f"ra_ref = {dm.schema_info(path='roman.meta.wcsinfo.ra_ref')['description']}")
print(f"dec_ref = {dm.schema_info(path='roman.meta.wcsinfo.dec_ref')['description']}")
print(f"roll_ref = {dm.schema_info(path='roman.meta.wcsinfo.roll_ref')['description']}")

ra_ref = Right ascension of the reference pixel position on the
sky in units of degrees.

dec_ref = Declination of the reference pixel position on the sky
in units of degrees.

roll_ref = Position angle of the telescope V3 axis at the
reference pixel position measured from North to East. The
telescope V coordinate system is defined with its origin at the
vertex of the primary mirror. The +V3 axis is orthogonal to the
sunshield. See Roman-STScI-000143 "Description of the Roman SIAF
and Coordinate Frames" for more information.



The "reference pixel position" mentioned in these descriptions is located at the center of each WFI detector (each detector has its own WCS). We can edit these values to make the pipeline create a WCS solution that is slightly different from the original. Let's make a copy of the data (for comparison later) and apply a simple shift of 1 arcsecond in right ascension:

In [16]:
original_ra_ref = copy.copy(dm.meta.wcsinfo.ra_ref)
dm.meta.wcsinfo.ra_ref += (1 / 3600)

print(f'Original ra_ref = {original_ra_ref},\nUpdated ra_ref = {dm.meta.wcsinfo.ra_ref}')

Original ra_ref = 270.8719766359773,
Updated ra_ref = 270.8722544137551


Next, let's run AssignWcsStep on the data:

In [17]:
result = romancal.assign_wcs.AssignWcsStep.call(dm)

2026-07-22 21:51:11,353 - stpipe.step - INFO - AssignWcsStep instance created.


2026-07-22 21:51:11,541 - stpipe.step - INFO - Step AssignWcsStep running with args (<roman_datamodels.datamodels._datamodels.ImageModel object at 0x7f4ccd28fb00>,).


2026-07-22 21:51:11,543 - stpipe.step - INFO - Step AssignWcsStep parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: None
  output_ext: .asdf
  output_use_model: False
  output_use_index: True
  save_results: False
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  update_version: False


2026-07-22 21:51:11,544 - romancal.assign_wcs.assign_wcs_step - INFO - reftype, distortion


2026-07-22 21:51:11,547 - romancal.assign_wcs.assign_wcs_step - INFO - Using reference files: {'distortion': '/home/runner/crds_cache/references/roman/wfi/roman_wfi_distortion_0014.asdf'} for assign_wcs


2026-07-22 21:51:11,604 - stcal.alignment.util - INFO - Update S_REGION to POLYGON ICRS  270.934962807 -0.225734417 270.934905036 -0.102770948 270.809314973 -0.102466475 270.810037271 -0.225325334


2026-07-22 21:51:11,605 - romancal.assign_wcs.assign_wcs - INFO - S_REGION VALUES: POLYGON ICRS  270.934962807 -0.225734417 270.934905036 -0.102770948 270.809314973 -0.102466475 270.810037271 -0.225325334


2026-07-22 21:51:11,605 - romancal.assign_wcs.assign_wcs - INFO - Update S_REGION to POLYGON ICRS  270.934962807 -0.225734417 270.934905036 -0.102770948 270.809314973 -0.102466475 270.810037271 -0.225325334


2026-07-22 21:51:13,420 - romancal.stpipe.core - INFO - Results used CRDS context: roman_0058.pmap


2026-07-22 21:51:13,420 - stpipe.step - INFO - Step AssignWcsStep done


Let’s now verify whether the coordinate system has changed as expected by comparing the right ascension and declination of a given pixel in the two WCS reference systems. For simplicity, we’ll use the center of the L2 image, which corresponds to (x, y) = (2043.5, 2043.5) in 0-indexed pixels. The corresponding sky coordinates can be easily obtained using `astropy.coordinates.SkyCoord` objects:

In [18]:
# Get SkyCoord object for new position at center of detector
ra, dec = result.meta.wcs(2043.5, 2043.5)
result_coord = SkyCoord(ra=ra, dec=dec, unit='deg')
result_coord

# Get SkyCoord object for original position at center of detector
ra0, dec0 = original_wcs(2043.5, 2043.5)
original_coord = SkyCoord(ra=ra0, dec=dec0, unit='deg')
original_coord

# Compute the separation between the updated and original positions
result_coord.separation(original_coord)

<Angle 0.00027702 deg>

As we can see, the newly updated WCS is shifted by approximately 1 arcsecond relative to the WCS in the original L2 file. The offset is not exactly 1 arcsecond because the WFI pixel grid is slightly rotated with respect to the celestial coordinate system. In other words, the detector axes are not perfectly aligned with the vertical (declination) and horizontal (right ascension) directions on the sky.

As in our pipeline example above, we can also pass optional arguments to individual steps. This is useful if we want to use our own version, or an older version, of a reference file. 

## Reference Files

As mentioned above, several of the pipeline steps apply reference files to the data to correct for specific detectors' effects.
Common examples of reference files applied during the processing of imaging data include the **Bad Pixel Mask**, **Dark**, and **Flat**. More information on WFI reference file types can be found in the RDox article [CRDS for Reference Files](https://roman-docs.stsci.edu/data-handbook-home/accessing-wfi-data/crds-for-reference-files).

As you run the exposure pipeline, the most up-to-date reference files will be automatically selected for each step. However, if you would like to use a specific reference file, these can be retrieved through the `crds` Python API and the ELP run with those files (more on that later). Let's begin with how to access reference files from CRDS.

The notebook [Understanding CRDS and How to Select Calibration Reference files](crds_reference_files.ipynb) provides information on how to identify and download reference files by name. Once available we can override `RomanCal` to use these reference files.

## How to Override RomanCal with Local Reference Files

If you have a local reference file that you would like to use in RomanCal processing, either one you retrieved from CRDS or one that you made yourself, then you can supply that when you run the ELP as an optional argument. The new reference file you indicate will override the CRDS best reference file selection from the server. Each reference file type has its own override parameter name (e.g., `override_mask`) and must be passed as arguments to the relevant step. Let's look at an example now where we run the pipeline and override the mask and flat files. First, we need new reference files to use. For this example, let's run `crds.getreferences()` and get new files for a different set of observation parameters, and then we can use those to override the best references. Our example is generally a bad idea, but for real applications you will likely already have a file on your disk storage that you want to use. Our example is to show how to override the reference file selection.

For our example, we will process a WFI01 L1 file with the ELP, but we will override the mask and flat files to be those from WFI04. Again, **this is not recommended** but simply designed to show how to override the reference file selection used by the pipeline if you have your own calibration reference files you want to use.

**Note:** In `romancal` version 0.20.2 (August 2025), there is a bug when overriding reference files. Please make sure to disable the source catalog and tweakreg steps (as shown below) when overriding reference files to bypass this bug. This bug is already fixed in the next `romancal` version (0.21.0; November 2025), which is undergoing testing and validation.

In [19]:
meta = {'ROMAN.META.INSTRUMENT.NAME': 'WFI',
        'ROMAN.META.INSTRUMENT.DETECTOR': 'WFI04',
        'ROMAN.META.INSTRUMENT.OPTICAL_ELEMENT': 'F158',
        'ROMAN.META.EXPOSURE.TYPE': 'WFI_IMAGE',
        'ROMAN.META.EXPOSURE.START_TIME': '2026-01-01 00:00:00'
       }

ref_files = crds.getreferences(meta, reftypes=['dark'], observatory='roman')
print(ref_files)

{'dark': '/home/runner/crds_cache/references/roman/wfi/roman_wfi_dark_0628.asdf'}


In [20]:
result = ExposurePipeline.call(dm_l1, save_results=False, steps={
                'source_catalog': {'skip': True},
                'tweakreg': {'skip': True},
                'dark_current': {'override_dark': ref_files['dark']}})

2026-07-22 21:51:13,460 - stpipe.step - INFO - ExposurePipeline instance created.


2026-07-22 21:51:13,461 - stpipe.step - INFO - DQInitStep instance created.


2026-07-22 21:51:13,462 - stpipe.step - INFO - SaturationStep instance created.


2026-07-22 21:51:13,463 - stpipe.step - INFO - RefPixStep instance created.


2026-07-22 21:51:13,464 - stpipe.step - INFO - DarkDecayStep instance created.


...

2026-07-22 21:52:34,413 - stpipe.step - INFO - Step skipped.


2026-07-22 21:52:35,360 - stpipe.step - INFO - Step tweakreg running with args (<romancal.datamodels.library.ModelLibrary object at 0x7f4cdc1ac050>,).


2026-07-22 21:52:35,361 - stpipe.step - INFO - Step skipped.


2026-07-22 21:52:36,124 - romancal.pipeline.exposure_pipeline - INFO - Roman exposure calibration pipeline ending...


2026-07-22 21:52:36,886 - stpipe.step - INFO - Step ExposurePipeline done


Now let's take a look at our `result` variable, which is our L2 datamodel in memory. Specifically, we can check the `meta.ref_file` section to see the names of the reference files used:

In [21]:
for k, v in result.meta.ref_file.items():
    print(f'{k} = {v}')

apcorr = ?
area = ?
crds = {'context': 'roman_0058.pmap', 'version': '13.2.7'}
dark = /home/runner/crds_cache/references/roman/wfi/roman_wfi_dark_0628.asdf
darkdecaysignal = crds://roman_wfi_darkdecaysignal_0001.asdf
distortion = crds://roman_wfi_distortion_0014.asdf
epsf = ?
flat = crds://roman_wfi_flat_0175.asdf
gain = crds://roman_wfi_gain_0022.asdf
integralnonlinearity = crds://roman_wfi_integralnonlinearity_0003.asdf
inverselinearity = crds://roman_wfi_inverselinearity_0042.asdf
linearity = crds://roman_wfi_linearity_0039.asdf
mask = crds://roman_wfi_mask_0030.asdf
photom = crds://roman_wfi_photom_0040.asdf
readnoise = crds://roman_wfi_readnoise_0024.asdf
refpix = crds://roman_wfi_refpix_0013.asdf
saturation = crds://roman_wfi_saturation_0031.asdf


Indeed, we see that many of the files used came from CRDS (note the file names begin with "crds://") whereas the flat and mask files contain a local path to the files we selected.

## About this Notebook
**Author:** T. Desjardins & R. Diaz

**Updated On:** 2026-07-06

<table width="100%" style="border:none; border-collapse:collapse;">

  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>